<a href="https://colab.research.google.com/github/psept77-star/Emp_training_risk_prediction/blob/main/TrainingCompliance_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Predict which employees are at risk of missing mandatory training deadlines**

In [1]:
# import liabraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

# load data set
df = pd.read_csv('/content/sample_data/Trainingprojectdata.csv')


In [11]:
# Feature selection
df['Due_Date'] = pd.to_datetime(df['Due_Date'])
df['Completion_Date'] = pd.to_datetime(df['Completion_Date'])
df['Days_to_Complete'] = (df['Due_Date'] - pd.Timestamp.today()).dt.days
df['Training_Pending'] = df['Completion_Date'].isna().astype(int)

# Drop rows where 'Risk_Flag' is NaN to ensure a clean target variable
df.dropna(subset=['Risk_Flag'], inplace=True)

# Encode Categorical variable
# Identify which of the target categorical columns actually exist in the DataFrame
intended_categorical_cols = ['Department', 'Training_Name']
existing_categorical_cols = [col for col in intended_categorical_cols if col in df.columns]

# Only proceed with get_dummies if there are existing columns to encode
if existing_categorical_cols:
    df = pd.get_dummies(df, columns=existing_categorical_cols, drop_first=True)
else:
    # If no intended categorical columns were found, a warning will be implicitly handled in the spoken response.
    # No code action needed here as df remains unchanged for these columns.
    pass

# Define Featues and target
# Dynamically create a list of columns to drop, ensuring they exist in the DataFrame
cols_to_drop = ['Employee_ID', 'Risk_Flag', 'Due_Date', 'Completion_Date']

# Filter cols_to_drop to only include columns present in df to avoid errors if a column was already dropped or never existed
effective_cols_to_drop = [col for col in cols_to_drop if col in df.columns]
x = df.drop(columns=effective_cols_to_drop, errors='ignore')
y = df['Risk_Flag']

In [12]:
# Train test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


# Model selection
models = {
    'Random Forest': RandomForestClassifier(),
    'Logistic Regression': LogisticRegression(),
    'Support Vector Machine': SVC(),
    'Decision Tree': DecisionTreeClassifier(),
    'AdaBoost': AdaBoostClassifier()
}


In [13]:
for name, model in models.items():
    print(f"Training {name}..")
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    print(f"\nClassification Report for {name}: ")
    print(classification_report(y_test, y_pred))

Training Random Forest..

Classification Report for Random Forest: 
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00         4
         1.0       1.00      1.00      1.00         2

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

Training Logistic Regression..

Classification Report for Logistic Regression: 
              precision    recall  f1-score   support

         0.0       1.00      0.50      0.67         4
         1.0       0.50      1.00      0.67         2

    accuracy                           0.67         6
   macro avg       0.75      0.75      0.67         6
weighted avg       0.83      0.67      0.67         6

Training Support Vector Machine..

Classification Report for Support Vector Machine: 
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         4
   

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
# Evaluation
y_pred = model.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00         4
         1.0       1.00      1.00      1.00         2

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



In [15]:
# Predicting employee at risk using algorithm Random forest
rf_model = models['Random Forest']
y_pred_risk = rf_model.predict(x_test)

risk_predictions_df = pd.DataFrame({
    'Employee_ID': df.loc[x_test.index, 'Employee_ID'],
    'Predicted_Risk_Flag': y_pred_risk
})

employees_at_risk = risk_predictions_df[risk_predictions_df['Predicted_Risk_Flag'] == 1.0]

print("Employees at risk:")
if not employees_at_risk.empty:
    display(employees_at_risk[['Employee_ID']])
    print(f"Total employees at risk: {len(employees_at_risk)}")
else:
    print("No employees predicted to be at risk in the test set.")

Employees at risk:


,Employee_ID
8,EMP009
16,EMP017


Total employees at risk: 2


In [18]:
# Predicting employee at risk using algorithm Decision Tree
dt_model = models['Decision Tree']
y_pred_risk = dt_model.predict(x_test)

risk_predictions_df = pd.DataFrame({
    'Employee_ID': df.loc[x_test.index, 'Employee_ID'],
    'Predicted_Risk_Flag': y_pred_risk
})

employees_at_risk = risk_predictions_df[risk_predictions_df['Predicted_Risk_Flag'] == 1.0]

print("Employees at risk:")
if not employees_at_risk.empty:
    display(employees_at_risk[['Employee_ID']])
    print(f"Total employees at risk: {len(employees_at_risk)}")
else:
    print("No employees predicted to be at risk in the test set.")

Employees at risk:


,Employee_ID
8,EMP009
16,EMP017


Total employees at risk: 2


In [19]:
# Predicting employee at risk using algorithm Adaboost
ada_model = models['AdaBoost']
y_pred_risk = ada_model.predict(x_test)

risk_predictions_df = pd.DataFrame({
    'Employee_ID': df.loc[x_test.index, 'Employee_ID'],
    'Predicted_Risk_Flag': y_pred_risk
})

employees_at_risk = risk_predictions_df[risk_predictions_df['Predicted_Risk_Flag'] == 1.0]

print("Employees at risk:")
if not employees_at_risk.empty:
    display(employees_at_risk[['Employee_ID']])
    print(f"Total employees at risk: {len(employees_at_risk)}")
else:
    print("No employees predicted to be at risk in the test set.")

Employees at risk:


,Employee_ID
8,EMP009
16,EMP017


Total employees at risk: 2
